# 无评分实验：提示词工程


## 1. 介绍

欢迎来到提示词工程无评分实验！在本实验中，你将探索一些提示词技巧，帮助你让 LLM 更贴合你的需求。主要内容包括：


1. 学习如何让 LLM 生成特定输出，例如给句子打标签
2. 根据提示任务类型，使用不同参数调用 LLM
3. 让 LLM 在响应中返回特定对象类型，例如 JSON


### 1.1 导入库


# 目录
- [ 1 - 使用 LLM 进行文本分类](#1)
- [ 2 - 基于任务的参数设置](#2)
- [ 3 - 引导 LLM 输出特定对象](#3)


In [1]:
from utils import (
    generate_with_single_input, 
    generate_with_multiple_input, 
    generate_params_dict
)

<a id='1'></a>
## 1 - 使用 LLM 进行文本分类

语言模型（LLM）一个有趣且实用的应用，是将其转化为文本分类器。通过清晰的指令，你可以引导 LLM 按情感、任务类型等维度对文本进行分类。由于分类器通常输出固定标签（例如正向为 1、负向为 0），你需要构造提示词以尽量降低 LLM 产生意外输出的概率。除了设计稳健的提示词，也要在代码中加入校验，避免后续流程预期接收 1 或 0，却收到 2 或“positive sentence”这类异常值。两者结合可以在保持灵活性的同时，获得更可靠的分类结果。

为了说明这一点，假设你正在为一家销售运动服饰和营养补剂的公司开发聊天机器人。目标是让 LLM 判断用户问题属于服饰还是营养方向。这样你就可以将请求路由到对应数据库进行检索。

1. 指令要精确。明确告诉模型你要它做什么、输出什么。
2. 添加示例。给出示例及预期结果。
3. 也可以加入边界示例，即你预期模型可能难以判断的样例。

In [2]:
def check_if_outfit_or_supplement(query):
    prompt = f"""
Determine the category of the following query as either "nutritional" or "outfit" related.
- Nutritional queries: These are related to nutrition products, such as whey protein, vitamins, supplements, dietary products, and health-related food and beverages.
  - Outfit queries: These pertain to clothing and fashion, including items like shirts, dresses, shoes, accessories, and jewelry.
Examples:

1. Query: “Where can I buy high-protein snacks?” Expected answer: Nutritional
2. Query: “Best shirt styles for summer 2023” Expected answer: Outfit
3. Query: “Are there any shoes designed for running?” Expected answer: Outfit
4. Query: “What multivitamins should I take daily?” Expected answer: Nutritional
5. Query: “Best weight loss products that are stylish” Expected answer: Nutritional
6. Query: “Athletic wear that boosts performance” Expected answer: Outfit 

Query: {query}

Instructions: Respond with “Nutritional” if the query pertains to nutritional products or “Outfit” if it pertains to clothing or fashion products.
Answer only one single word.
"""
    return prompt
    

In [ ]:
# 测试一个简单查询
query = "Give me the available vitamins supplement you have in your catalogue."
generate_with_single_input(check_if_outfit_or_supplement(query), max_tokens = 2)

{'role': 'assistant', 'content': 'Nutritional'}

下面我们在更大的样本集上测试。

In [ ]:
# ASCII 颜色代码
GREEN = '\033[92m'
RED = '\033[91m'
RESET = '\033[0m'

queries = [
    {"query": "Where can I buy whey protein?", "label": "Nutritional"},
    {"query": "Recommended vitamins for winter", "label": "Nutritional"},
    {"query": "Latest fashion for women's dresses", "label": "Outfit"},
    {"query": "Comfortable sneakers for daily use", "label": "Outfit"},
    {"query": "Best energy bars for athletes", "label": "Nutritional"},
    {"query": "Trendy accessories for men", "label": "Outfit"},
    {"query": "Low-carb diet food options", "label": "Nutritional"},
    {"query": "What supplements help with muscle recovery?", "label": "Nutritional"},
    {"query": "Casual wear that supports healthy living", "label": "Outfit"}
]

for item in queries:
    query = item["query"]
    prompt = check_if_outfit_or_supplement(query)
    expected_label = item["label"]
    response = generate_with_single_input(prompt, max_tokens = 2)
    result = response['content']
    
    # 根据比较结果选择颜色
    if result == expected_label:
        color = GREEN
    else:
        color = RED

    print(f"Query: {query}\nResult: {result}\nExpected: {color}{expected_label}{RESET}\n")
    

Query: Where can I buy whey protein?
Result: Nutritional
Expected: Nutritional

Query: Recommended vitamins for winter
Result: Nutritional
Expected: Nutritional

Query: Latest fashion for women's dresses
Result: Outfit
Expected: Outfit

Query: Comfortable sneakers for daily use
Result: Outfit
Expected: Outfit

Query: Best energy bars for athletes
Result: Nutritional
Expected: Nutritional

Query: Trendy accessories for men
Result: Outfit
Expected: Outfit

Query: Low-carb diet food options
Result: Nutritional
Expected: Nutritional

Query: What supplements help with muscle recovery?
Result: Nutritional
Expected: Nutritional

Query: Casual wear that supports healthy living
Result: Outfit
Expected: Outfit



<a id='2'></a>
## 2 - 基于任务的参数设置

在这一部分中，你将学习如何灵活调整与 LLM 的交互方式，从而根据任务类型控制其行为。这通常需要在让 LLM 作答前，先判断查询的任务属性。

在本练习中，我们将开发一个函数，把查询分类为技术类或创意类。完成分类后，就可以为不同任务类型应用不同参数。技术类查询通常更适合较低随机性，而创意类任务通常可以使用更高随机性。

In [ ]:
def decide_if_technical_or_creative(query):
    """
    判断给定查询更偏向创意类还是技术类。

    参数:
        query (str): 待评估的查询字符串。

    返回:
        str: 查询类型标签，取值为 'creative' 或 'technical'。

    该函数构造一个提示词，根据查询内容进行分类。
    创意类查询通常要求生成原创内容，技术类查询通常与文档或流程信息相关。
    通过调用 LLM，函数返回对应的类型标签。
    """
    
    PROMPT = f"""Decide if the following query is a creative query or a technical query.
    Creative queries ask you to create content, while technical queries are related to documentation or technical requests, like information about procedures.
    Answer only 'creative' or 'technical'.
    Query: {query}
    """
    result = generate_with_single_input(PROMPT)
    label = result['content']
    return label

In [6]:
queries = ["What is Pi-hole?", 
           "Suggest to me three places to visit in South America"]
for query in queries:
    label =decide_if_technical_or_creative(query)
    print(f"Query: {query}, label: {label}")

Query: What is Pi-hole?, label: technical
Query: Suggest to me three places to visit in South America, label: creative


In [ ]:
def answer_query(query):
    """
    处理查询并生成合适回答：先将查询分类为 'technical' 或 'creative'，
    再依据分类调整生成参数。

    参数:
        query (str): 需要回答的查询字符串。

    返回:
        str: 基于查询类型生成的 LLM 响应内容。

    函数首先调用 `decide_if_technical_or_creative` 判断查询性质。
    若为 'technical'，使用更精确、低随机性的参数；
    若为 'creative'，使用更具变化性的参数；
    若判断不明确，则使用中性参数。
    最后按对应参数生成响应并返回文本内容。
    """
    
    # 判断查询是 'technical' 还是 'creative'
    label = decide_if_technical_or_creative(query).lower()

    # 技术类查询参数（更精确、低随机性）
    if label == 'technical':
        kwargs = generate_params_dict(query, temperature=0, top_p=0.1)
    
    # 创意类查询参数（变化更大、随机性更高）
    elif label == 'creative':
        kwargs = generate_params_dict(query, temperature=1.1, top_p=0.4)

    # 若类型判断不明确，使用默认参数
    else:
        kwargs = generate_params_dict(query, temperature=0.5, top_p=0.5)
    
    # 基于查询类型和参数生成响应
    response = generate_with_single_input(**kwargs)
    
    # 提取并返回响应内容
    result = response['content']
    return result

In [8]:
queries = ["What is Pi-hole?", 
           "Suggest to me three places to visit in South America"]
for query in queries:
    result = answer_query(query)
    print(f"Query: {query}\nAnswer: {result}\n\n#######\n")

Query: What is Pi-hole?
Answer: **Pi-hole** is a free, open-source, **network-wide ad blocker** designed to run on a single device (typically a Raspberry Pi, though it works on any Linux server). It acts as a "sinkhole" for DNS queries, effectively blocking ads, trackers, and malware domains across your entire home network.

Here is a breakdown of how it works and why it is popular:

### How It Works
Instead of installing ad-blocking software on every single device (like your phone, laptop, or smart TV), Pi-hole sits at the gateway of your network.
1.  **DNS Interception**: When a device tries to load a website, it first asks the DNS server (like Google's `8.8.8.8`) for the IP address of that domain.
2.  **The Block**: Pi-hole intercepts this request. If the domain is known to serve ads or trackers, Pi-hole refuses to provide the IP address.
3.  **The Sinkhole**: Instead of the real IP, Pi-hole returns a fake IP address (usually `0.0.0.0`). The device then tries to connect to this fake

<a id='3'></a>
## 3 - 引导 LLM 输出特定对象

在这一部分中，你将学习如何让 LLM 按特定格式输出结果（例如 JSON），以便被其他应用直接使用。这在实际开发中非常关键，因为应用通常要求数据结构严格可控。

假设你正在做家庭自动化，希望创建个人助手来控制灯光和音响等设备。目标是把用户请求转换为家庭自动化服务器可理解的结构化格式。

在这个示例场景中，每个动作的格式是一个 JSON 结构，包含如下信息：

```json
{
  "room": "room where the action will occur",
  "object_id": "unique identifier of the targeted object",
  "object_name": "name of the object",
  "action": "action to be performed",
  "parameters": "dictionary containing action-specific parameters"
}
```

例如，要打开办公室灯并设置为黄色，你应向软件提供以下 JSON：

```json
{
  "room": "office",
  "object_id": "152",
  "object_name": "office_light",
  "action": "turn on",
  "parameters": {"color": "yellow"}
}
```


### 3.1 传统方式

我们先从传统方式开始：构建一个详细提示词并传给 LLM。

下面这个提示词示例结构较完整。注意：在提示词中嵌入 JSON 格式时，需要正确处理 `f-string` 语法以避免报错。

**注意**：提示词工程通常需要大量实验。比较不同提示词的输出、发现潜在问题并持续迭代优化，是非常正常且必要的过程。

In [9]:
def generate_system_call(command):
    PROMPT = f"""
You are an assistant program that converts natural language commands into structured JSON for controlling smart home devices. The JSON should conform to a specific format describing the device, action, and parameters. Here's how you can do it:

**Available Devices and Actions:**

1. **Light**
   - Actions: "turn on", "turn off"
   - Parameters: color, intensity (percentage)

2. **Automatic Lock**
   - Actions: "lock", "unlock"
   - Parameters: None

3. **Sound System (Speaker)**
   - Actions: "play", "pause", "stop", "set volume"
   - Parameters: volume (integer), track (string), playlist_style (string)

4. **TV**
   - Actions: "turn on", "turn off", "change channel", "adjust volume"
   - Parameters: channel (string), volume (integer)

5. **Air Conditioner**
   - Actions: "turn on", "turn off", "set temperature", "adjust fan speed"
   - Parameters: temperature (integer), fan_speed (low/medium/high)

**Rooms and Devices:**
- **Office**
  - Lights: "office_light_1" (ID: 123), "office_light_2" (ID: 321)
  - Automatic Lock: "office_door_lock" (ID: 111)

- **Living Room**
  - Light: "living_room_light" (ID: 222)
  - Speaker: "living_room_speaker" (ID: 223)
  - Air Conditioner: "living_room_airconditioner" (ID: 556)

- **Kitchen**
  - Light: "kitchen_light" (ID: 333)

- **Bedroom**
  - Light: "bedroom_light" (ID: 444)
  - TV: "bedroom_tv" (ID: 445)

- **Bathroom**
  - Light: "bathroom_light" (ID: 555)

**Task:**
Convert the following natural language command into the structured JSON format based on the available devices:

**Input Examples:**

1. "Turn on the office light with ID 123 with blue color and 50% intensity."
   - JSON:
     [
     {{
       "room": "office",
       "object_id": "123",
       "object_name": "office_light_1",
       "action": "turn on",
       "parameters": {{"color": "blue", "intensity": "50%"}}
     }}
     ]

2. "Lock the office door."
   - JSON:
   [
     {{
       "room": "office",
       "object_id": "111",
       "object_name": "office_door_lock",
       "action": "lock",
       "parameters": {{}}
     }}
    ]

2. "Make my living room a cheerful place"
   - JSON:
   [
     {{
       "room": "living_room",
       "object_id": "222",
       "object_name": "living_room_light",
       "action": "turn on",
       "parameters": {{'intensity': '80%', 'color':'yellow'}}
     }},
     {{
       "room": "living_room",
       "object_id": "223",
       "object_name": "living_room_speaker",
       "action": "turn on",
       "parameters": {{'volume': '100', 'playlist_style':'party'}}
     }},
     
   ]

**Note:**
- Ensure that each JSON object correctly maps the natural command to the appropriate device and action using the listed device ID.
- Use the object ID to differentiate between devices when the room contains multiple similar items.
- You can add more than one parameter in the parameters dictionary.

Using this information, translate the following command into JSON: "{command}". Output a list with all the necessary JSONs. 
Always output a list even if there is only one command to be applied, do not output anything else but the desired structure.
"""
    kwargs = generate_params_dict(PROMPT, temperature=0.4, top_p=0.1)
    result = generate_with_single_input(**kwargs)
    return result['content']

In [10]:
print(generate_system_call("Play a chill playlist very loud"))

[
  {
    "room": "living_room",
    "object_id": "223",
    "object_name": "living_room_speaker",
    "action": "play",
    "parameters": {
      "playlist_style": "chill",
      "volume": "100"
    }
  }
]


In [11]:
print(generate_system_call("I'm tired today, please make my living room a very cozy ambient, it is really cold today too."))

[
  {
    "room": "living_room",
    "object_id": "222",
    "object_name": "living_room_light",
    "action": "turn on",
    "parameters": {
      "color": "warm white",
      "intensity": "40%"
    }
  },
  {
    "room": "living_room",
    "object_id": "556",
    "object_name": "living_room_airconditioner",
    "action": "set temperature",
    "parameters": {
      "temperature": "24",
      "fan_speed": "low"
    }
  }
]


### 3.2 使用 LLM 的结构化输出参数

你可以使用 [Pydantic](https://docs.pydantic.dev/latest/) 来约束 LLM 输出 JSON，并帮助校验数据结构，从而尽量确保输出始终是 JSON！

下面看一个示例！

In [ ]:
from pydantic import BaseModel, validator, conint, Field
from typing import Literal, Union, Optional, List
import json

# 定义输出结构
class VoiceNote(BaseModel):
    title: str = Field(description="A title for the voice note")
    summary: str = Field(description="A short one sentence summary of the voice note.")
    actionItems: list[str] = Field(
        description="A list of action items from the voice note"
    )

In [13]:
transcript = (
        "Good morning! It's 7:00 AM, and I'm just waking up. Today is going to be a busy day, "
        "so let's get started. First, I need to make a quick breakfast. I think I'll have some "
        "scrambled eggs and toast with a cup of coffee. While I'm cooking, I'll also check my "
        "emails to see if there's anything urgent."
    )


messages=[
            {
                "role": "system",
                "content": "The following is a voice message transcript. Only answer in JSON.",
            },
            {
                "role": "user",
                "content": transcript,
            },
        ]

response_format={
            "type": "json_schema",
            "schema": VoiceNote.model_json_schema(),
        }

result = generate_with_multiple_input(messages, response_format = response_format)
result_json = json.loads(result['content'])
print(json.dumps(result_json, indent=2))

{
  "title": "Morning Routine and Breakfast Plans",
  "summary": "At 7:00 AM, the speaker is starting a busy day, planning to eat scrambled eggs, toast, and coffee, and checking emails while cooking.",
  "actionItems": [
    "Make scrambled eggs",
    "Prepare toast",
    "Brew a cup of coffee",
    "Check emails for urgent items"
  ]
}


继续加油！你已完成提示词工程无评分实验！